# Notebook 06A: AlphaFold Structure Preparation

## AI-Based Identification of Novel Drug Targets for Depression

### Overview

This notebook prepares protein structural data required for the Graph Neural
Network modelling stage of the project.

AlphaFold protein structures are retrieved for the depression-associated protein
targets identified during the earlier stages of the pipeline. The downloaded
structures provide three-dimensional information required to construct
residue-level protein graphs for graph-based deep learning.

The protein targets are mapped using their UniProt identifiers, and available
AlphaFold Protein Structure Database models are downloaded and organised within
the project data structure.

In addition to downloading structural files, this notebook performs validation
checks to determine structure availability and generates metadata files that
record the relationship between protein identifiers and their corresponding
structural models.

The prepared structural dataset will be used in Notebook 07 for residue-level
graph construction and Graph Neural Network development.


### Dataset

The input dataset for this notebook is:

`protein_info.csv`

The dataset contains:

- UniProt identifiers
- Protein names
- Protein sequences
- Protein length information

These identifiers are used to retrieve corresponding AlphaFold protein
structures.


### Output Files

This notebook generates the following outputs:

`data/raw/structures/`

Contains **174 downloaded AlphaFold protein structure files** in PDB format.
These structural models correspond to available protein targets identified by
their UniProt identifiers and are used for residue-level graph construction in
the Graph Neural Network stage.

`data/raw/`

`structures_metadata.csv`

Contains information linking the 200 UniProt identifiers with their
corresponding structural files and download status. This metadata records which
protein structures were successfully retrieved and which structures were not
available.

`missing_structures.csv`

Contains the 26 protein targets for which AlphaFold structures were not
available. These records are maintained for transparency and reproducibility of
the structural data preparation process.


### Project Workflow

This notebook represents the supporting preparation stage between feature
integration and Graph Neural Network development:

1. Load protein information dataset.
2. Extract UniProt identifiers.
3. Retrieve AlphaFold protein structures.
4. Store available PDB structural files.
5. Validate structure availability.
6. Generate structural metadata.
7. Prepare structural inputs for residue-level graph construction.


### Objectives

The objectives of this notebook are:

1. Retrieve AlphaFold protein structures for target proteins.
2. Organise structural files within the project directory.
3. Validate protein structure availability.
4. Generate metadata for reproducible structural analysis.
5. Prepare structural information required for Graph Neural Network modelling.

In [2]:
import os

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = (
    "/content/drive/MyDrive/Dissertation/"
    "Depression-Drug-Target-AI"
)

RAW_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw"
)

PROCESSED_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

STRUCTURES_PATH = os.path.join(
    RAW_DATA_PATH,
    "structures"
)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Structures path:", STRUCTURES_PATH)
print("Structures folder exists:", os.path.exists(STRUCTURES_PATH))

Mounted at /content/drive
Project root: /content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI
Raw data path: /content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/raw
Processed data path: /content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/processed
Structures path: /content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/raw/structures
Structures folder exists: False


In [4]:
import os

for root, dirs, files in os.walk(PROJECT_ROOT):
    pdb_files = [
        f for f in files
        if f.lower().endswith(".pdb")
    ]

    if pdb_files:
        print("\nPDB folder found:")
        print(root)
        print("Number of PDB files:", len(pdb_files))
        print("First 20 files:", pdb_files[:20])

In [5]:
DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data"
)

print("Contents of data folder:")
print(os.listdir(DATA_PATH))

print("\nRaw folder:")
print(os.listdir(RAW_DATA_PATH))

print("\nProcessed folder:")
print(os.listdir(PROCESSED_DATA_PATH))

Contents of data folder:
['raw', 'processed', 'external']

Raw folder:
['drug_target_interactions.csv', 'protein_info.csv', 'depression_genes.csv']

Processed folder:
['top20_depression_genes.csv', 'protein_sequences.fasta', 'protein_cleaned.csv', 'clean_drug_target_interactions.csv', 'clean_drug_target_interactions.pkl', 'random_forest_evaluation.csv', 'random_forest_cross_validation.csv', 'random_forest_model.joblib', 'protein_embeddings.npy', 'protein_embedding_metadata.csv', 'integrated_features.npy', 'integrated_labels.npy', 'integrated_metadata.csv']


In [6]:
import pandas as pd

protein_info_path = os.path.join(
    RAW_DATA_PATH,
    "protein_info.csv"
)

protein_info = pd.read_csv(
    protein_info_path
)

print(protein_info.head())

print("\nColumns:")
print(protein_info.columns.tolist())

print("\nNumber of proteins:")
print(len(protein_info))

        ensembl_id  uniprot_id                            protein_name  \
0  ENSG00000149295  A0A1Y8EK52                                     NaN   
1  ENSG00000102468  A0A7P0PKG8         5-hydroxytryptamine receptor 2A   
2  ENSG00000022355  A0A1B0GU82                                     NaN   
3  ENSG00000183454  A0A6Q8PGD2                      Glutamate receptor   
4  ENSG00000108576      P31645  Sodium-dependent serotonin transporter   

                                            sequence  length  
0                                            MDPLNLS       7  
1  MHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMP...     308  
2  MRKSPGLSDCLWAWILLLSTLTGRRTSRKSHQNTDLYEVNEGKDVA...      70  
3  MLKIMQDYDWHVFSLVTTIFPGYREFISFVKTTVDNSFVGWDMQNV...    1307  
4  METTPLNSQKQLSACEDGEDCQENGVLQKVVPTPGDKVESGQISNG...     630  

Columns:
['ensembl_id', 'uniprot_id', 'protein_name', 'sequence', 'length']

Number of proteins:
200


In [7]:
import requests

uniprot_id = "P31645"

url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

response = requests.get(url)

print(response.status_code)

if response.status_code == 200:
    print(response.json()[0]["pdbUrl"])
else:
    print("Structure not available")

200
https://alphafold.ebi.ac.uk/files/AF-P31645-F1-model_v6.pdb


In [8]:
# 1. Set Up Project Directories

from google.colab import drive
import os

drive.mount("/content/drive")


PROJECT_ROOT = (
    "/content/drive/MyDrive/Dissertation/"
    "Depression-Drug-Target-AI"
)


RAW_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw"
)


STRUCTURES_PATH = os.path.join(
    RAW_DATA_PATH,
    "structures"
)


os.makedirs(
    STRUCTURES_PATH,
    exist_ok=True
)


print("Project path configured successfully.")
print("Structures path:")
print(STRUCTURES_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project path configured successfully.
Structures path:
/content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/raw/structures


In [9]:
# 2. Import Required Libraries

import pandas as pd
import requests
import os
from tqdm import tqdm

In [10]:
# 3. Load Protein Information Dataset


protein_info_path = os.path.join(
    RAW_DATA_PATH,
    "protein_info.csv"
)


protein_info = pd.read_csv(
    protein_info_path
)


print(
    "Number of proteins:",
    len(protein_info)
)


protein_info.head()

Number of proteins: 200


,ensembl_id,uniprot_id,protein_name,sequence,length
0,ENSG00000149295,A0A1Y8EK52,NaN,MDPLNLS,7
1,ENSG00000102468,A0A7P0PKG8,5-hydroxytryptamine receptor 2A,MHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMP...,308
2,ENSG00000022355,A0A1B0GU82,NaN,MRKSPGLSDCLWAWILLLSTLTGRRTSRKSHQNTDLYEVNEGKDVA...,70
3,ENSG00000183454,A0A6Q8PGD2,Glutamate receptor,MLKIMQDYDWHVFSLVTTIFPGYREFISFVKTTVDNSFVGWDMQNV...,1307
4,ENSG00000108576,P31645,Sodium-dependent serotonin transporter,METTPLNSQKQLSACEDGEDCQENGVLQKVVPTPGDKVESGQISNG...,630


In [11]:
# 4. AlphaFold Structure Download Function


def download_alphafold_structure(uniprot_id, save_directory):

    filename = (
        f"AF-{uniprot_id}-F1-model_v6.pdb"
    )


    save_path = os.path.join(
        save_directory,
        filename
    )


    url = (
        f"https://alphafold.ebi.ac.uk/files/"
        f"AF-{uniprot_id}-F1-model_v6.pdb"
    )


    if os.path.exists(save_path):

        return {
            "uniprot_id": uniprot_id,
            "structure_file": filename,
            "status": "already_exists"
        }


    response = requests.get(
        url,
        timeout=30
    )


    if response.status_code == 200:

        with open(
            save_path,
            "w"
        ) as file:

            file.write(
                response.text
            )


        return {
            "uniprot_id": uniprot_id,
            "structure_file": filename,
            "status": "downloaded"
        }


    else:

        return {
            "uniprot_id": uniprot_id,
            "structure_file": None,
            "status": "not_available"
        }

In [12]:
# 5. Download AlphaFold Structures


structure_records = []


for uniprot_id in tqdm(
    protein_info["uniprot_id"],
    desc="Downloading AlphaFold structures"
):

    result = download_alphafold_structure(
        uniprot_id,
        STRUCTURES_PATH
    )


    structure_records.append(
        result
    )


print(
    "Structure download completed."
)

Structure download completed.


In [13]:
# 6. Create Structure Metadata


structures_metadata = pd.DataFrame(
    structure_records
)


structures_metadata.head()

,uniprot_id,structure_file,status
0,A0A1Y8EK52,None,not_available
1,A0A7P0PKG8,AF-A0A7P0PKG8-F1-model_v6.pdb,downloaded
2,A0A1B0GU82,AF-A0A1B0GU82-F1-model_v6.pdb,downloaded
3,A0A6Q8PGD2,None,not_available
4,P31645,AF-P31645-F1-model_v6.pdb,downloaded


In [14]:
# 7. Check Structure Availability


print(
    structures_metadata["status"]
    .value_counts()
)

status
downloaded       174
not_available     26
Name: count, dtype: int64


In [16]:
# 8. Save Structures Metadata


metadata_path = os.path.join(
    RAW_DATA_PATH,
    "structures_metadata.csv"
)


structures_metadata.to_csv(
    metadata_path,
    index=False
)


print(
    "Metadata saved successfully:"
)

print(
    metadata_path
)

Metadata saved successfully:
/content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/raw/structures_metadata.csv


In [17]:
# 9. Verify Downloaded Structures


pdb_files = [
    f for f in os.listdir(STRUCTURES_PATH)
    if f.endswith(".pdb")
]


print(
    "Number of PDB files:",
    len(pdb_files)
)


print(
    "\nFirst 10 structures:"
)

print(
    pdb_files[:10]
)

Number of PDB files: 174

First 10 structures:
['AF-A0A7P0PKG8-F1-model_v6.pdb', 'AF-A0A1B0GU82-F1-model_v6.pdb', 'AF-P31645-F1-model_v6.pdb', 'AF-Q5ZGX3-F1-model_v6.pdb', 'AF-A0AAQ5BGN6-F1-model_v6.pdb', 'AF-A0A8V8TLH3-F1-model_v6.pdb', 'AF-A0A1W2PPS4-F1-model_v6.pdb', 'AF-H3BM11-F1-model_v6.pdb', 'AF-H0YAS3-F1-model_v6.pdb', 'AF-A0A1B0GUX7-F1-model_v6.pdb']


In [18]:
# 10. Final Structure Preparation Summary


print(
    "Total proteins:",
    len(protein_info)
)


print(
    "Downloaded structures:",
    len(pdb_files)
)


print(
    "Metadata records:",
    len(structures_metadata)
)

Total proteins: 200
Downloaded structures: 174
Metadata records: 200


In [19]:
# Check proteins without available structures

missing_structures = structures_metadata[
    structures_metadata["status"] == "not_available"
]


print(
    "Missing structures:",
    len(missing_structures)
)


missing_structures.head()

Missing structures: 26


,uniprot_id,structure_file,status
0,A0A1Y8EK52,None,not_available
3,A0A6Q8PGD2,None,not_available
10,A0A6Q8PGN4,None,not_available
22,A0A1W2PR72,None,not_available
25,H0YJU6,None,not_available


In [20]:
missing_path = os.path.join(
    RAW_DATA_PATH,
    "missing_structures.csv"
)


missing_structures.to_csv(
    missing_path,
    index=False
)


print(
    "Missing structure report saved:"
)

print(
    missing_path
)

Missing structure report saved:
/content/drive/MyDrive/Dissertation/Depression-Drug-Target-AI/data/raw/missing_structures.csv


In [21]:
# Check structure coverage in interaction dataset

interaction_path = os.path.join(
    RAW_DATA_PATH,
    "drug_target_interactions.csv"
)

interactions = pd.read_csv(
    interaction_path
)


available_ids = set(
    structures_metadata[
        structures_metadata["status"] != "not_available"
    ]["uniprot_id"]
)


coverage = interactions["uniprot_id"].isin(
    available_ids
)


print(
    "Total interactions:",
    len(interactions)
)


print(
    "Interactions with structures:",
    coverage.sum()
)


print(
    "Coverage percentage:",
    round(
        (coverage.sum() / len(interactions)) * 100,
        2
    ),
    "%"
)

Total interactions: 4164
Interactions with structures: 4164
Coverage percentage: 100.0 %


## Summary of Findings

This notebook successfully prepared protein structural data required for the
Graph Neural Network modelling stage.

AlphaFold structures were retrieved using UniProt identifiers from the protein
information dataset. Out of the 200 target proteins, 174 AlphaFold protein
structures were successfully downloaded, while 26 proteins did not have
available structural models.

The generated structure metadata provides a record of structural availability
and links protein identifiers with their corresponding PDB files.

All 4164 drug-target interaction records were successfully mapped to proteins
with available structural information, providing complete structural coverage
for the interaction dataset.

The prepared AlphaFold structures and metadata files provide the required
inputs for residue-level graph construction and Graph Neural Network-based
drug-target interaction prediction in the next stage of the project.